In [1]:
from nba_api.stats.endpoints import playercareerstats
from nba_api.stats.static import players
import pandas as pd

# Find LeBron James in the NBA database
lebron = players.find_players_by_full_name("LeBron James")[0]
print("Player found:", lebron)

Player found: {'id': 2544, 'full_name': 'LeBron James', 'first_name': 'LeBron', 'last_name': 'James', 'is_active': True}


In [2]:
# Use LeBron's ID to fetch his entire career stats
career = playercareerstats.PlayerCareerStats(player_id=2544)

# The NBA API returns data in a special format
# .get_data_frames() converts it into a pandas DataFrame
# Think of it as converting raw data into a clean spreadsheet
df = career.get_data_frames()[0]

# Show the shape of the data
# This tells us how many rows and columns we have
print("Shape:", df.shape)

# Show the column names
print("\nColumns:", list(df.columns))

Shape: (23, 27)

Columns: ['PLAYER_ID', 'SEASON_ID', 'LEAGUE_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'PLAYER_AGE', 'GP', 'GS', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS']


In [3]:
# Show the last 5 seasons of LeBron's career
# tail(5) means "give me the last 5 rows"
# Think of it like scrolling to the bottom of a spreadsheet

df_clean = df[['SEASON_ID', 'TEAM_ABBREVIATION', 'GP', 'PTS', 'AST', 'REB', 'FG_PCT']].tail(5)
print(df_clean.to_string())

   SEASON_ID TEAM_ABBREVIATION  GP   PTS  AST  REB  FG_PCT
18   2021-22               LAL  56  1695  349  459   0.524
19   2022-23               LAL  55  1590  375  457   0.500
20   2023-24               LAL  71  1822  589  518   0.540
21   2024-25               LAL  70  1710  575  546   0.513
22   2025-26               LAL  57  1184  403  346   0.512


In [4]:
# Calculate points per game for each season
# This is your first data transformation in pandas

# Create a new column called PPG (points per game)
# We divide total points by games played for each row
# pandas does this for every row automatically, no loop needed
df['PPG'] = df['PTS'] / df['GP']
df['APG'] = df['AST'] / df['GP']
df['RPG'] = df['REB'] / df['GP']

# Round to 1 decimal place to make it readable
df['PPG'] = df['PPG'].round(1)
df['APG'] = df['APG'].round(1)
df['RPG'] = df['RPG'].round(1)

# Show the last 5 seasons with our new calculated columns
result = df[['SEASON_ID', 'GP', 'PPG', 'APG', 'RPG', 'FG_PCT']].tail(5)
print(result.to_string())

   SEASON_ID  GP   PPG  APG  RPG  FG_PCT
18   2021-22  56  30.3  6.2  8.2   0.524
19   2022-23  55  28.9  6.8  8.3   0.500
20   2023-24  71  25.7  8.3  7.3   0.540
21   2024-25  70  24.4  8.2  7.8   0.513
22   2025-26  57  20.8  7.1  6.1   0.512


In [5]:
# LeBron is 40 years old and still putting up 20+ PPG
# Let's find his career best and worst seasons
# and see his full scoring journey

print("=== LeBron James Career Scoring Journey ===\n")

# Find his best scoring season
best_season = df.loc[df['PPG'].idxmax()]
print(f"Best scoring season: {best_season['SEASON_ID']} with {best_season['PPG']} PPG")

# Find his current form vs career average
career_avg = df['PPG'].mean().round(1)
current_ppg = df.iloc[-1]['PPG']
diff = round(current_ppg - career_avg, 1)

print(f"Career average PPG: {career_avg}")
print(f"Current season PPG: {current_ppg}")
print(f"Difference from career avg: {diff} PPG")

# Find seasons where he scored 25+ PPG
elite_seasons = df[df['PPG'] >= 25.0]
print(f"\nSeasons with 25+ PPG: {len(elite_seasons)} out of {len(df)} total seasons")

=== LeBron James Career Scoring Journey ===

Best scoring season: 2005-06 with 31.4 PPG
Career average PPG: 26.7
Current season PPG: 20.8
Difference from career avg: -5.9 PPG

Seasons with 25+ PPG: 20 out of 23 total seasons


In [6]:
from nba_api.stats.endpoints import playergamelog
import time

# Fetch LeBron's game by game log for this season
# 2024-25 is the season ID format the NBA API uses
print("Fetching LeBron's game logs... this takes a few seconds")

# We add time.sleep to avoid hitting the API too fast
# The NBA API has rate limiting, meaning if you call it
# too many times too quickly it will block you temporarily
time.sleep(1)

gamelog = playergamelog.PlayerGameLog(
    player_id=2544,
    season='2024-25'
)

games = gamelog.get_data_frames()[0]

print(f"Games fetched: {len(games)}")
print(f"\nColumns available: {list(games.columns)}")

Fetching LeBron's game logs... this takes a few seconds
Games fetched: 70

Columns available: ['SEASON_ID', 'Player_ID', 'Game_ID', 'GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS', 'PLUS_MINUS', 'VIDEO_AVAILABLE']


In [7]:
# Let's look at his last 10 games
# The data comes in reverse chronological order
# so the most recent games are at the top

recent = games[['GAME_DATE', 'MATCHUP', 'WL', 'PTS', 'AST', 'REB', 'FG_PCT', 'PLUS_MINUS']].head(10)
print("=== LeBron's Last 10 Games ===\n")
print(recent.to_string())

# Calculate his averages over the last 10 games
print("\n=== Last 10 Game Averages ===")
print(f"PPG:  {games['PTS'].head(10).mean().round(1)}")
print(f"APG:  {games['AST'].head(10).mean().round(1)}")
print(f"RPG:  {games['REB'].head(10).mean().round(1)}")
print(f"FG%:  {(games['FG_PCT'].head(10).mean() * 100).round(1)}%")

# Find his best and worst games this season
best_game = games.loc[games['PTS'].idxmax()]
worst_game = games.loc[games['PTS'].idxmin()]

print(f"\nBest game this season:  {best_game['PTS']} pts vs {best_game['MATCHUP']} on {best_game['GAME_DATE']}")
print(f"Worst game this season: {worst_game['PTS']} pts vs {worst_game['MATCHUP']} on {worst_game['GAME_DATE']}")

=== LeBron's Last 10 Games ===

      GAME_DATE      MATCHUP WL  PTS  AST  REB  FG_PCT  PLUS_MINUS
0  Apr 11, 2025  LAL vs. HOU  W   14    8    4   0.545           5
1  Apr 09, 2025    LAL @ DAL  W   27    3    7   0.550          18
2  Apr 08, 2025    LAL @ OKC  L   28    3    7   0.421         -23
3  Apr 06, 2025    LAL @ OKC  W   19    7    3   0.563           7
4  Apr 04, 2025  LAL vs. NOP  W   27    8    0   0.529           7
5  Apr 03, 2025  LAL vs. GSW  L   33    9    5   0.667          -1
6  Mar 31, 2025  LAL vs. HOU  W   16    4    8   0.417           9
7  Mar 29, 2025    LAL @ MEM  W   25    8    6   0.476          -1
8  Mar 27, 2025    LAL @ CHI  L   17   12    5   0.400         -10
9  Mar 26, 2025    LAL @ IND  W   13    7   13   0.333           5

=== Last 10 Game Averages ===
PPG:  21.9
APG:  6.9
RPG:  5.8
FG%:  49.0%

Best game this season:  42 pts vs LAL vs. GSW on Feb 06, 2025
Worst game this season: 10 pts vs LAL @ MIN on Dec 02, 2024


In [8]:
# Home vs Away analysis
# Look at the MATCHUP column
# "LAL vs. XXX" means HOME game
# "LAL @ XXX" means AWAY game

# Create a new column that identifies home or away
# If the matchup contains "vs." it is a home game
games['HOME_AWAY'] = games['MATCHUP'].apply(
    lambda x: 'HOME' if 'vs.' in x else 'AWAY'
)

# Group by home/away and calculate averages
# groupby is like a pivot table in Excel
# it groups rows by a category and calculates stats for each group
home_away_stats = games.groupby('HOME_AWAY')[['PTS', 'AST', 'REB', 'FG_PCT']].mean().round(2)

print("=== LeBron Home vs Away This Season ===\n")
print(home_away_stats.to_string())

# Count wins at home vs away
win_rate = games.groupby('HOME_AWAY')['WL'].apply(
    lambda x: f"{(x == 'W').sum()}W - {(x == 'L').sum()}L"
)

print("\n=== Win/Loss Record ===")
print(win_rate.to_string())

=== LeBron Home vs Away This Season ===

             PTS   AST   REB  FG_PCT
HOME_AWAY                           
AWAY       23.97  8.25  7.56    0.49
HOME       24.91  8.18  8.06    0.53

=== Win/Loss Record ===
HOME_AWAY
AWAY    19W - 17L
HOME     25W - 9L


In [9]:
# Rest days analysis
# Convert GAME_DATE to a proper date format so we can do math on it
games['GAME_DATE'] = pd.to_datetime(games['GAME_DATE'])

# Sort by date oldest to newest so we can calculate gaps
games_sorted = games.sort_values('GAME_DATE').reset_index(drop=True)

# Calculate days between each game
# diff() finds the difference between consecutive rows
# dt.days converts the time difference to just the number of days
games_sorted['DAYS_REST'] = games_sorted['GAME_DATE'].diff().dt.days

# Fill the first game with 3 days rest as default
games_sorted['DAYS_REST'] = games_sorted['DAYS_REST'].fillna(3)

# Group by rest days and see how scoring changes
rest_analysis = games_sorted.groupby('DAYS_REST')[['PTS', 'FG_PCT']].agg({
    'PTS': ['mean', 'count'],
    'FG_PCT': 'mean'
}).round(2)

print("=== LeBron Performance by Rest Days ===\n")
print(rest_analysis.to_string())

=== LeBron Performance by Rest Days ===

             PTS       FG_PCT
            mean count   mean
DAYS_REST                    
1.0        25.36    11   0.51
2.0        24.74    43   0.51
3.0        24.67     9   0.54
4.0        21.50     2   0.48
6.0        20.50     2   0.58
7.0        26.00     1   0.46
9.0        18.00     1   0.41
14.0       17.00     1   0.44


In [1]:
from nba_api.stats.endpoints import leaguedashteamstats
import time

time.sleep(1)

print("Fetching all NBA team defensive stats for 2024-25 season...")

# Fetch defensive stats for all 30 NBA teams
team_stats = leaguedashteamstats.LeagueDashTeamStats(
    season='2024-25',
    measure_type_detailed_defense='Defense'
)

# Convert to DataFrame
defense_df = team_stats.get_data_frames()[0]

# Show what columns we have
print(f"Teams fetched: {len(defense_df)}")
print(f"Columns: {list(defense_df.columns)}")

Fetching all NBA team defensive stats for 2024-25 season...
Teams fetched: 30
Columns: ['TEAM_ID', 'TEAM_NAME', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'DEF_RATING', 'DREB', 'DREB_PCT', 'STL', 'BLK', 'OPP_PTS_OFF_TOV', 'OPP_PTS_2ND_CHANCE', 'OPP_PTS_FB', 'OPP_PTS_PAINT', 'GP_RANK', 'W_RANK', 'L_RANK', 'W_PCT_RANK', 'MIN_RANK', 'DEF_RATING_RANK', 'DREB_RANK', 'DREB_PCT_RANK', 'STL_RANK', 'BLK_RANK', 'OPP_PTS_OFF_TOV_RANK', 'OPP_PTS_2ND_CHANCE_RANK', 'OPP_PTS_FB_RANK', 'OPP_PTS_PAINT_RANK']


In [2]:
# Let's see the best and worst defenses in the NBA right now
# Sort by DEF_RATING ascending because LOWER = BETTER defense

defense_ranked = defense_df[['TEAM_NAME', 'DEF_RATING', 'W', 'L', 'W_PCT']].sort_values(
    'DEF_RATING', ascending=True
).reset_index(drop=True)

print("=== NBA Team Defensive Rankings 2024-25 ===")
print("(Lower DEF_RATING = Better Defense)\n")

print("TOP 5 BEST DEFENSES:")
print(defense_ranked.head(5).to_string())

print("\nBOTTOM 5 WORST DEFENSES:")
print(defense_ranked.tail(5).to_string())

=== NBA Team Defensive Rankings 2024-25 ===
(Lower DEF_RATING = Better Defense)

TOP 5 BEST DEFENSES:
               TEAM_NAME  DEF_RATING   W   L  W_PCT
0  Oklahoma City Thunder       106.6  68  14  0.829
1          Orlando Magic       109.1  41  41  0.500
2            LA Clippers       109.4  50  32  0.610
3         Boston Celtics       110.1  61  21  0.744
4        Houston Rockets       110.3  52  30  0.634

BOTTOM 5 WORST DEFENSES:
               TEAM_NAME  DEF_RATING   W   L  W_PCT
25    Philadelphia 76ers       117.3  24  58  0.293
26          Phoenix Suns       117.7  36  46  0.439
27    Washington Wizards       118.0  18  64  0.220
28  New Orleans Pelicans       119.1  21  61  0.256
29             Utah Jazz       119.4  17  65  0.207


In [4]:
from nba_api.stats.endpoints import playergamelog
import pandas as pd
import time

time.sleep(1)

# Re-fetch LeBron's game logs
gamelog = playergamelog.PlayerGameLog(
    player_id=2544,
    season='2024-25'
)

games = gamelog.get_data_frames()[0]

# Re-add the HOME_AWAY column we created yesterday
games['HOME_AWAY'] = games['MATCHUP'].apply(
    lambda x: 'HOME' if 'vs.' in x else 'AWAY'
)

# Re-convert date column
games['GAME_DATE'] = pd.to_datetime(games['GAME_DATE'])

print(f"Games reloaded: {len(games)}")
print("Ready to continue!")

Games reloaded: 70
Ready to continue!


In [5]:
# Now let's connect LeBron's game logs with opponent defensive ratings
# This is called a JOIN in data science
# We are combining two datasets based on a common column

# First create a lookup dictionary of team name to defensive rating
# A dictionary is like a phone book: team name -> defensive rating
defense_lookup = dict(zip(defense_df['TEAM_NAME'], defense_df['DEF_RATING']))

# The matchup column looks like "LAL vs. OKC" or "LAL @ BOS"
# We need to extract just the opponent abbreviation
# Then match it to the full team name

# Let's first see what opponent abbreviations look like in the game log
print("Sample matchups from LeBron's game log:")
print(games['MATCHUP'].head(10).to_string())
print("\nSample team names from defense data:")
print(defense_df['TEAM_NAME'].head(10).to_string())

Sample matchups from LeBron's game log:
0    LAL vs. HOU
1      LAL @ DAL
2      LAL @ OKC
3      LAL @ OKC
4    LAL vs. NOP
5    LAL vs. GSW
6    LAL vs. HOU
7      LAL @ MEM
8      LAL @ CHI
9      LAL @ IND

Sample team names from defense data:
0            Atlanta Hawks
1           Boston Celtics
2            Brooklyn Nets
3        Charlotte Hornets
4            Chicago Bulls
5      Cleveland Cavaliers
6         Dallas Mavericks
7           Denver Nuggets
8          Detroit Pistons
9    Golden State Warriors


In [6]:
# Build a mapping from team abbreviation to full team name
# This is a common data cleaning task in sports analytics
# Two datasets rarely use the same format for team names

abbrev_to_name = {
    'ATL': 'Atlanta Hawks',
    'BOS': 'Boston Celtics',
    'BKN': 'Brooklyn Nets',
    'CHA': 'Charlotte Hornets',
    'CHI': 'Chicago Bulls',
    'CLE': 'Cleveland Cavaliers',
    'DAL': 'Dallas Mavericks',
    'DEN': 'Denver Nuggets',
    'DET': 'Detroit Pistons',
    'GSW': 'Golden State Warriors',
    'HOU': 'Houston Rockets',
    'IND': 'Indiana Pacers',
    'LAC': 'LA Clippers',
    'LAL': 'Los Angeles Lakers',
    'MEM': 'Memphis Grizzlies',
    'MIA': 'Miami Heat',
    'MIL': 'Milwaukee Bucks',
    'MIN': 'Minnesota Timberwolves',
    'NOP': 'New Orleans Pelicans',
    'NYK': 'New York Knicks',
    'OKC': 'Oklahoma City Thunder',
    'ORL': 'Orlando Magic',
    'PHI': 'Philadelphia 76ers',
    'PHX': 'Phoenix Suns',
    'POR': 'Portland Trail Blazers',
    'SAC': 'Sacramento Kings',
    'SAS': 'San Antonio Spurs',
    'TOR': 'Toronto Raptors',
    'UTA': 'Utah Jazz',
    'WAS': 'Washington Wizards'
}

# Now extract the opponent abbreviation from the matchup string
# "LAL vs. HOU" -> opponent is "HOU"
# "LAL @ OKC"   -> opponent is "OKC"
# Split by space and take the last part

def extract_opponent(matchup):
    # Split the matchup string by space
    # "LAL vs. HOU".split() gives ['LAL', 'vs.', 'HOU']
    # [-1] takes the last element which is always the opponent
    return matchup.split()[-1]

games['OPPONENT_ABB'] = games['MATCHUP'].apply(extract_opponent)

# Now map abbreviation to full name
games['OPPONENT_NAME'] = games['OPPONENT_ABB'].map(abbrev_to_name)

# Now add the defensive rating for each opponent
defense_lookup = dict(zip(defense_df['TEAM_NAME'], defense_df['DEF_RATING']))
games['OPP_DEF_RATING'] = games['OPPONENT_NAME'].map(defense_lookup)

# Show the result
print("=== LeBron Games with Opponent Defensive Rating ===\n")
result = games[['GAME_DATE', 'MATCHUP', 'OPPONENT_NAME', 'OPP_DEF_RATING', 'PTS']].head(10)
print(result.to_string())

=== LeBron Games with Opponent Defensive Rating ===

   GAME_DATE      MATCHUP          OPPONENT_NAME  OPP_DEF_RATING  PTS
0 2025-04-11  LAL vs. HOU        Houston Rockets           110.3   14
1 2025-04-09    LAL @ DAL       Dallas Mavericks           115.0   27
2 2025-04-08    LAL @ OKC  Oklahoma City Thunder           106.6   28
3 2025-04-06    LAL @ OKC  Oklahoma City Thunder           106.6   19
4 2025-04-04  LAL vs. NOP   New Orleans Pelicans           119.1   27
5 2025-04-03  LAL vs. GSW  Golden State Warriors           111.0   33
6 2025-03-31  LAL vs. HOU        Houston Rockets           110.3   16
7 2025-03-29    LAL @ MEM      Memphis Grizzlies           112.6   25
8 2025-03-27    LAL @ CHI          Chicago Bulls           114.8   17
9 2025-03-26    LAL @ IND         Indiana Pacers           113.3   13


In [7]:
# Let's prove the relationship between opponent defense and LeBron's scoring
# We will split games into "strong defense" and "weak defense" categories

# The NBA average defensive rating this season is around 113
# Below 113 = strong defense, above 113 = weak defense
avg_def_rating = defense_df['DEF_RATING'].mean()
print(f"NBA average defensive rating: {avg_def_rating.round(1)}")

# Split games into two groups
strong_defense = games[games['OPP_DEF_RATING'] < avg_def_rating]
weak_defense = games[games['OPP_DEF_RATING'] >= avg_def_rating]

print(f"\nGames vs strong defense (DEF_RATING < {avg_def_rating.round(1)}): {len(strong_defense)}")
print(f"Games vs weak defense (DEF_RATING >= {avg_def_rating.round(1)}): {len(weak_defense)}")

print(f"\n=== LeBron vs Strong Defense ===")
print(f"PPG:  {strong_defense['PTS'].mean().round(1)}")
print(f"FG%:  {(strong_defense['FG_PCT'].mean() * 100).round(1)}%")
print(f"Win%: {(strong_defense['WL'].eq('W').mean() * 100).round(1)}%")

print(f"\n=== LeBron vs Weak Defense ===")
print(f"PPG:  {weak_defense['PTS'].mean().round(1)}")
print(f"FG%:  {(weak_defense['FG_PCT'].mean() * 100).round(1)}%")
print(f"Win%: {(weak_defense['WL'].eq('W').mean() * 100).round(1)}%")

NBA average defensive rating: 113.7

Games vs strong defense (DEF_RATING < 113.7): 36
Games vs weak defense (DEF_RATING >= 113.7): 34

=== LeBron vs Strong Defense ===
PPG:  24.5
FG%:  50.6%
Win%: 58.3%

=== LeBron vs Weak Defense ===
PPG:  24.4
FG%:  51.6%
Win%: 67.6%


In [8]:
# Feature 4: Recent Form
# The idea is simple: if LeBron scored 35, 32, 28 in his last 3 games
# he is hot and likely to score well in the next game too
# If he scored 12, 15, 14 he is cold and likely to struggle

# Sort games oldest to newest for rolling calculations
games_sorted = games.sort_values('GAME_DATE').reset_index(drop=True)

# rolling(5) means "look at the last 5 rows"
# .mean() calculates the average of those 5 rows
# .shift(1) shifts the result forward by 1 row
# so we are using the PREVIOUS 5 games to predict the CURRENT game
# without this shift we would be cheating by including the current game

games_sorted['FORM_5'] = games_sorted['PTS'].rolling(5).mean().shift(1).round(1)
games_sorted['FORM_3'] = games_sorted['PTS'].rolling(3).mean().shift(1).round(1)

# Show the result
print("=== LeBron Recent Form vs Actual Scoring ===\n")
result = games_sorted[['GAME_DATE', 'MATCHUP', 'FORM_5', 'FORM_3', 'PTS']].tail(15)
print(result.to_string())

=== LeBron Recent Form vs Actual Scoring ===

    GAME_DATE      MATCHUP  FORM_5  FORM_3  PTS
55 2025-03-04  LAL vs. NOP    26.0    26.0   34
56 2025-03-06  LAL vs. NYK    27.8    26.3   31
57 2025-03-08    LAL @ BOS    28.6    27.3   22
58 2025-03-22  LAL vs. CHI    26.4    29.0   17
59 2025-03-24    LAL @ ORL    24.2    23.3   24
60 2025-03-26    LAL @ IND    25.6    21.0   13
61 2025-03-27    LAL @ CHI    21.4    18.0   17
62 2025-03-29    LAL @ MEM    18.6    18.0   25
63 2025-03-31  LAL vs. HOU    19.2    18.3   16
64 2025-04-03  LAL vs. GSW    19.0    19.3   33
65 2025-04-04  LAL vs. NOP    20.8    24.7   27
66 2025-04-06    LAL @ OKC    23.6    25.3   19
67 2025-04-08    LAL @ OKC    24.0    26.3   28
68 2025-04-09    LAL @ DAL    24.6    24.7   27
69 2025-04-11  LAL vs. HOU    26.8    24.7   14


In [9]:
# Feature 5: Win Loss Streak
# Is LeBron on a winning streak or losing streak?
# Teams on winning streaks have higher morale and confidence
# Teams on losing streaks often show lower individual performance

# Convert W/L to 1/0 for math
games_sorted['WIN'] = games_sorted['WL'].apply(lambda x: 1 if x == 'W' else 0)

# Calculate rolling win rate over last 5 games
# This tells us what percentage of the last 5 games were wins
games_sorted['WIN_RATE_5'] = games_sorted['WIN'].rolling(5).mean().shift(1).round(2)

# Now let's see if winning streaks correlate with better scoring
high_confidence = games_sorted[games_sorted['WIN_RATE_5'] >= 0.6]
low_confidence = games_sorted[games_sorted['WIN_RATE_5'] <= 0.4]

print("=== Performance on Winning Streaks (60%+ win rate last 5) ===")
print(f"Games:  {len(high_confidence)}")
print(f"PPG:    {high_confidence['PTS'].mean().round(1)}")
print(f"FG%:    {(high_confidence['FG_PCT'].mean() * 100).round(1)}%")

print("\n=== Performance on Losing Streaks (40% or less win rate last 5) ===")
print(f"Games:  {len(low_confidence)}")
print(f"PPG:    {low_confidence['PTS'].mean().round(1)}")
print(f"FG%:    {(low_confidence['FG_PCT'].mean() * 100).round(1)}%")

print("\n=== All 5 ML Features Discovered ===")
print("Feature 1: Home or Away")
print("Feature 2: Days of rest")
print("Feature 3: Opponent defensive rating")
print("Feature 4: Recent form (last 5 game average)")
print("Feature 5: Win rate last 5 games (team momentum)")

=== Performance on Winning Streaks (60%+ win rate last 5) ===
Games:  44
PPG:    25.8
FG%:    52.5%

=== Performance on Losing Streaks (40% or less win rate last 5) ===
Games:  21
PPG:    22.2
FG%:    48.7%

=== All 5 ML Features Discovered ===
Feature 1: Home or Away
Feature 2: Days of rest
Feature 3: Opponent defensive rating
Feature 4: Recent form (last 5 game average)
Feature 5: Win rate last 5 games (team momentum)
